# Anatomy of an HTTP request and response 🌐📚

**Member 8 · Monday · Library Loans API**

By the end, you can trace a request from a client to a server and back, name the parts, and draft endpoints for **books, members and loans**.

**How to teach:** Read the short explanations, ask the prediction questions, and run each code cell with **Shift + Enter**. The examples use Python's standard library only. The final quiz is a separate HTML file.

## 1. The journey: client → server → client

Think of a **client** as a customer asking a library for information. A browser, mobile app or Python program can be a client. A **server** is a program that receives the request and sends a response.

**Journey:** client prepares a request → request travels to the server → server handles it → server sends a response → client reads the result.

**Endpoint:** a method and path used to access a particular API operation. For example, `GET /books` asks to list books. HTTP is the set of rules used to exchange these messages; HTTPS is HTTP protected in transit with TLS.

## 2. A request: what are we asking for?

```http
GET /books?available=true HTTP/1.1
Host: library.example
Accept: application/json
```

- **Method:** `GET` means we want to read data.
- **Path:** `/books` identifies the resource.
- **Query string:** `?available=true` asks for a filtered list.
- **HTTP version:** `HTTP/1.1` is part of this example's request line.
- **Headers:** `Host` identifies the host; `Accept` says what response format the client accepts.
- **Body:** optional data sent with a request, often JSON for `POST` or `PATCH`. This `GET` example has no body.

The full example URL would be `https://library.example/books?available=true`. The host above is fictional; we won't send it a real request.

In [1]:
# Inspect the parts of a URL without contacting a server.
from urllib.parse import parse_qs, urlparse

url = "https://library.example/books?available=true"
parts = urlparse(url)

print("Scheme:", parts.scheme)
print("Host:", parts.netloc)
print("Path:", parts.path)
print("Query:", parse_qs(parts.query))

Scheme: https
Host: library.example
Path: /books
Query: {'available': ['true']}


## 3. Common HTTP methods

| Method | Usual purpose | Example |
|---|---|---|
| `GET` | Read data | Get a list of books |
| `POST` | Create something or request an action | Create a loan |
| `PATCH` | Partially update data | Change a book title |
| `DELETE` | Delete a resource | Delete a member |

**Ask:** Which method would you use to read a book? To create a loan? HTTP defines method semantics, but the API's documentation describes exactly what each endpoint does.

In [2]:
# Draft a practical create-book request as Python data.
import json

new_book = {"title": "Dune", "author": "Frank Herbert"}
request_body = json.dumps(new_book)

print("Method: POST")
print("Path: /books")
print("Content-Type: application/json")
print("Body:", request_body)

Method: POST
Path: /books
Content-Type: application/json
Body: {"title": "Dune", "author": "Frank Herbert"}


## 4. A response: what did the server send back?

```http
HTTP/1.1 200 OK
Content-Type: application/json

[{"id": 1, "title": "Dune"}]
```

- **Status line:** HTTP version, status code (`200`) and reason phrase (`OK`).
- **Headers:** metadata; `Content-Type` tells the client the format of the response body.
- **Body:** the returned data, often JSON. Some responses have no body, for example `204 No Content`.

Common codes: **200** success, **201** created, **204** success with no content, **400** invalid request, **404** resource not found, **500** server error. An API may use other appropriate codes too.

In [3]:
# A client can decode a JSON response body into Python data.
response_body = '[{"id": 1, "title": "Dune"}]'
returned_books = json.loads(response_body)

print("Number of books:", len(returned_books))
print("First title:", returned_books[0]["title"])

Number of books: 1
First title: Dune


## 5. Watch a real request and response on your computer

The next cell starts a **temporary local server**, sends it a `GET` and a `POST`, and shuts it down. Look for the method, path, query string, request body, response status, header and response body.

This is a small teaching demo, **not** a full API: it doesn't save books. It uses a loopback connection on your computer; no external website is contacted.

In [2]:
import json
from http.server import BaseHTTPRequestHandler, HTTPServer
from threading import Thread
from urllib.parse import parse_qs, urlparse
from urllib.request import Request, urlopen


class LibraryDemo(BaseHTTPRequestHandler):
    def log_message(self, format, *args):
        pass  # Keep the output easy to read.

    def send_json(self, status, data):
        body = json.dumps(data).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        url = urlparse(self.path)
        print("Server received:", self.command, url.path,
              "query =", parse_qs(url.query))
        if url.path == "/books":
            self.send_json(200, [{"id": 1, "title": "Dune"}])
        else:
            self.send_json(404, {"error": "Not found"})

    def do_POST(self):
        if self.path != "/books":
            self.send_json(404, {"error": "Not found"})
            return
        length = int(self.headers.get("Content-Length", 0))
        data = json.loads(self.rfile.read(length).decode("utf-8"))
        print("Server received:", self.command, self.path, "body =", data)
        self.send_json(201, {"id": 2, "title": data["title"]})

server = HTTPServer(("127.0.0.1", 0), LibraryDemo)
thread = Thread(target=server.serve_forever, daemon=True)
thread.start()
base_url = f"http://127.0.0.1:{server.server_port}"

try:
    # Client asks for books.
    get_request = Request(base_url + "/books?available=true",
                          headers={"Accept": "application/json"})
    with urlopen(get_request) as response:
        print("Client got:", response.status,
              response.headers["Content-Type"], response.read().decode())

    # Client sends a new book as JSON.
    body = json.dumps({"title": "Matilda"}).encode("utf-8")
    post_request = Request(base_url + "/books", data=body, method="POST",
                           headers={"Content-Type": "application/json"})
    with urlopen(post_request) as response:
        print("Client got:", response.status,
              response.headers["Content-Type"], response.read().decode())
finally:
    server.shutdown()
    server.server_close()
    thread.join()

Server received: GET /books query = {'available': ['true']}
Client got: 200 application/json [{"id": 1, "title": "Dune"}]
Server received: POST /books body = {'title': 'Matilda'}
Client got: 201 application/json {"id": 2, "title": "Matilda"}


## 6. Draft endpoints for the library

These are **proposed** routes for the team's API, not endpoints already deployed. `{book_id}`, `{member_id}` and `{loan_id}` are placeholders for real IDs. A collection path such as `/books` represents multiple books; `/books/1` identifies one book.

| Method | Path | Purpose | Example success |
|---|---|---|---|
| GET | `/books` | List books; optionally filter using a query string | 200 |
| GET | `/books/{book_id}` | Get one book | 200 |
| POST | `/books` | Add a book | 201 |
| PATCH | `/books/{book_id}` | Update part of a book | 200 |
| DELETE | `/books/{book_id}` | Delete a book | 204 |
| GET | `/members` | List members | 200 |
| GET | `/members/{member_id}` | Get one member | 200 |
| POST | `/members` | Add a member | 201 |
| PATCH | `/members/{member_id}` | Update part of a member | 200 |
| DELETE | `/members/{member_id}` | Delete a member | 204 |
| GET | `/loans` | List loans; optionally filter, e.g. by member | 200 |
| GET | `/loans/{loan_id}` | Get one loan | 200 |
| POST | `/loans` | Borrow a book by creating a loan | 201 |
| POST | `/loans/{loan_id}/return` | Return a borrowed book | 200 |

The team should agree on validation, permissions and errors. For example, reject a loan if the book is unavailable; don't assume deletion is allowed when there are active loans.

In [ ]:
# Practical example: prepare a borrow request.
loan = {"book_id": 1, "member_id": 7}

print("POST /loans")
print("Content-Type: application/json")
print("Body:", json.dumps(loan))
print("Expected success: 201 Created with the new loan")

## 7. Practice: trace the journey

A member with ID `7` wants to borrow book `1`.

1. Who is the client? Who is the server?
2. Which method and path would create the loan?
3. What information belongs in the request's JSON body?
4. What status code could indicate successful creation?
5. Name a response header and explain what it tells the client.

Now try this function: it creates the body the client could send. This function does **not** send a request.

In [ ]:
def make_loan_body(book_id, member_id):
    """Create a JSON request body for a proposed loan."""
    loan = {"book_id": book_id, "member_id": member_id}
    return json.dumps(loan)

body = make_loan_body(1, 7)
print("Request body:", body)

# Simple checks: both IDs should be in the JSON.
parsed = json.loads(body)
assert parsed["book_id"] == 1
assert parsed["member_id"] == 7
print("Checks passed!")

### Suggested answers

1. The client is the member's app/browser; the server is the Library Loans API.
2. `POST /loans`.
3. The book and member IDs, for example `{"book_id": 1, "member_id": 7}`.
4. `201 Created`.
5. `Content-Type: application/json` tells the client the response body is JSON.

## 8. Quick recap

**Request:** method + path/query + headers + optional body.  
**Response:** status + headers + optional body.  
**Flow:** client → server → client.  
**Endpoint draft:** choose a useful method and path for each resource or action.

Open the separate quiz HTML file in a browser to try the 20-question, 10-minute knowledge check.